In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv("../dataset/adult_cleaned.csv")

X = df.drop("income", axis=1)

y = (
    df["income"]
    .astype(str)
    .str.strip()
    .str.replace(".", "", regex=False)
    .eq(">50K")
    .astype(int)
)

print("Dataset loaded:", df.shape)
print("Gender groups:")
print(X["sex"].value_counts())

Dataset loaded: (45194, 15)
Gender groups:
sex
Male      30509
Female    14685
Name: count, dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (36155, 14)
Testing data: (9039, 14)


In [4]:
training_data = X_train.copy()
training_data["income"] = y_train.values

group_counts = training_data.groupby(
    ["sex", "income"]
).size()

gender_counts = training_data.groupby("sex").size()
income_counts = training_data.groupby("income").size()

total = len(training_data)

weights = {}

for gender in gender_counts.index:
    for income in income_counts.index:

        actual_count = group_counts.get(
            (gender, income), 0
        )

        expected_count = (
            gender_counts[gender]
            * income_counts[income]
            / total
        )

        weights[(gender, income)] = (
            expected_count / actual_count
            if actual_count > 0 else 0
        )

sample_weights = [
    weights[(gender, income)]
    for gender, income in zip(
        X_train["sex"],
        y_train
    )
]

print("Reweighting completed!")
print("Sample weights:", sample_weights[:10])

Reweighting completed!
Sample weights: [np.float64(0.8480295571078988), np.float64(1.0944264767403513), np.float64(1.0944264767403513), np.float64(0.7925956758433931), np.float64(1.0944264767403513), np.float64(1.0944264767403513), np.float64(1.0944264767403513), np.float64(2.1906255751396193), np.float64(0.7925956758433931), np.float64(0.8480295571078988)]


In [5]:
categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_columns = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns
        ),
        (
            "numerical",
            "passthrough",
            numerical_columns
        )
    ]
)

reweighted_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                max_depth=8,
                random_state=42
            )
        )
    ]
)

reweighted_model.fit(
    X_train,
    y_train,
    classifier__sample_weight=sample_weights
)

print("Reweighted model trained successfully!")

C:\Users\Admin\AppData\Local\Temp\ipykernel_388\4109844822.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


Reweighted model trained successfully!


In [6]:
y_pred_reweighted = reweighted_model.predict(X_test)

print("Reweighted predictions completed!")
print(y_pred_reweighted[:10])

Reweighted predictions completed!
[0 0 0 0 0 0 1 1 0 0]


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("REWEIGHTED MODEL PERFORMANCE")
print("-----------------------------")

print("Accuracy:",
      round(accuracy_score(y_test, y_pred_reweighted), 4))

print("Precision:",
      round(precision_score(y_test, y_pred_reweighted, zero_division=0), 4))

print("Recall:",
      round(recall_score(y_test, y_pred_reweighted, zero_division=0), 4))

print("F1 Score:",
      round(f1_score(y_test, y_pred_reweighted, zero_division=0), 4))

REWEIGHTED MODEL PERFORMANCE
-----------------------------
Accuracy: 0.8401
Precision: 0.7322
Recall: 0.56
F1 Score: 0.6346
